# GCN for Attention-Fused Patient Embeddings

This notebook trains a two-layer sparse Graph Convolutional Network (GCN) on the attention-fused 64-dimensional patient embeddings and saved patient-similarity graphs.

- Training labels update the model.
- Validation AUPRC selects the checkpoint; validation F1 selects the threshold.
- Test labels are used only for the final evaluation.
- Feature scaling is fit on the training embeddings only.


## 1. Imports and project configuration

In [15]:
from __future__ import annotations

import copy
import json
import random
from pathlib import Path
from typing import Any

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    roc_auc_score,
)

# Find project root from the notebook's current working directory.
_start = Path.cwd().resolve()
_candidates = [_start, *_start.parents]
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "checkpoints" / "fusion").exists() and (p / "processed").exists()),
    _start,
)

EMBEDDING_DIR = PROJECT_ROOT / "checkpoints" / "fusion" / "attention_embeddings"
GRAPH_DIR = PROJECT_ROOT / "checkpoints" / "fusion" / "patient_graphs_knn"
OUTPUT_DIR = PROJECT_ROOT / "checkpoints" / "fusion" / "gnn_attention"

SEED = 42
MAX_EPOCHS = 300
PATIENCE = 20
HIDDEN_DIM = 128
DROPOUT = 0.30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 5e-4

print(f"Current directory: {Path.cwd()}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Embedding dir:     {EMBEDDING_DIR}")
print(f"Graph dir:         {GRAPH_DIR}")
print(f"Output dir:        {OUTPUT_DIR}")

if not EMBEDDING_DIR.exists():
    raise FileNotFoundError(f"Embedding directory not found: {EMBEDDING_DIR}")
if not GRAPH_DIR.exists():
    raise FileNotFoundError(f"Graph directory not found: {GRAPH_DIR}")


Current directory: /Users/aryandwivedi/Desktop/Project Hail Minor/src/GNN
Project root:      /Users/aryandwivedi/Desktop/Project Hail Minor
Embedding dir:     /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/attention_embeddings
Graph dir:         /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/patient_graphs_knn
Output dir:        /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_attention


## 2. Reproducibility and helper functions

These helpers set random seeds and handle patient IDs consistently when aligning the embedding rows with graph nodes.


In [16]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        try:
            torch.mps.manual_seed(seed)
        except Exception:
            pass


def first_key(npz: Any, candidates: tuple[str, ...], required: bool = True):
    for key in candidates:
        if key in npz.files:
            return npz[key]
    if required:
        raise KeyError(f"Could not find any of {candidates}; available keys: {npz.files}")
    return None


def id_key(value: Any) -> str:
    """Normalize common numeric/string patient IDs for matching."""
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    return str(value).strip()


## 3. Load embeddings and align patient order

Each embedding file should contain the 64D embeddings, patient IDs, and binary labels. If `nodes_{split}.npz` includes patient IDs, the embeddings are reordered to match the graph node order.


In [17]:
def load_embedding_file(split: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    path = EMBEDDING_DIR / f"attention_embeddings_{split}.npz"
    if not path.exists():
        raise FileNotFoundError(f"Embedding file not found: {path}")
    with np.load(path, allow_pickle=True) as data:
        x = first_key(data, ("embeddings", "fused_embeddings", "features", "X"))
        ids = first_key(data, ("patient_ids", "ids", "pids"))
        y = first_key(data, ("labels", "y", "target"))
    x = np.asarray(x, dtype=np.float32)
    ids = np.asarray(ids).reshape(-1)
    y = np.asarray(y).reshape(-1).astype(np.int64)
    if x.ndim != 2 or len(x) != len(ids) or len(y) != len(ids):
        raise ValueError(f"Bad shapes in {path}: embeddings={x.shape}, IDs={ids.shape}, labels={y.shape}")
    if not np.isfinite(x).all():
        raise ValueError(f"Non-finite values found in embeddings: {path}")
    labels = set(np.unique(y).tolist())
    if not labels.issubset({0, 1}):
        raise ValueError(f"Expected binary labels 0/1 in {path}, found {labels}")
    return x, ids, y


def load_graph_ids(split: str) -> np.ndarray | None:
    """Return graph node IDs when the graph metadata includes them."""
    path = GRAPH_DIR / f"nodes_{split}.npz"
    if not path.exists():
        return None
    with np.load(path, allow_pickle=True) as data:
        ids = first_key(data, ("patient_ids", "ids", "pids", "node_ids"), required=False)
    return None if ids is None else np.asarray(ids).reshape(-1)


def align_to_graph_order(split: str, x: np.ndarray, ids: np.ndarray, y: np.ndarray):
    graph_ids = load_graph_ids(split)
    if graph_ids is None:
        print(f"{split}: no graph node IDs found; assuming embedding-file order matches graph order.")
        return x, ids, y
    if len(graph_ids) != len(ids):
        raise ValueError(f"{split}: graph node IDs ({len(graph_ids)}) and embeddings ({len(ids)}) differ.")
    lookup = {id_key(pid): i for i, pid in enumerate(ids)}
    if len(lookup) != len(ids):
        raise ValueError(f"Duplicate patient IDs in {split} embedding file.")
    missing = [pid for pid in graph_ids if id_key(pid) not in lookup]
    if missing:
        raise ValueError(f"{split}: {len(missing)} graph IDs are absent from embeddings; examples: {missing[:5]}")
    order = np.asarray([lookup[id_key(pid)] for pid in graph_ids], dtype=np.int64)
    print(f"{split}: embeddings aligned to graph-node patient order.")
    return x[order], graph_ids, y[order]


## 4. Load and normalize adjacency matrices

The code uses a saved normalized adjacency matrix when available. Otherwise it symmetrizes the raw graph, adds self-loops, and applies symmetric GCN normalization: `D^(-1/2) (A + I) D^(-1/2)`.


In [18]:
import warnings
warnings.filterwarnings("ignore", message=".*Sparse invariant checks.*")
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torch.sparse')

def load_adjacency(split: str) -> sp.coo_matrix:
    norm_path = GRAPH_DIR / f"adjacency_norm_{split}.npz"
    raw_path = GRAPH_DIR / f"adjacency_{split}.npz"
    path = norm_path if norm_path.exists() else raw_path
    if not path.exists():
        raise FileNotFoundError(f"No adjacency for split '{split}'. Checked {norm_path} and {raw_path}")
    try:
        adj = sp.load_npz(path).astype(np.float32).tocsr()
    except Exception as exc:
        raise ValueError(f"Could not read {path} as a SciPy sparse .npz file.") from exc
    if adj.shape[0] != adj.shape[1]:
        raise ValueError(f"Adjacency must be square, found {adj.shape} in {path}")
    if norm_path.exists():
        print(f"{split}: using pre-normalized adjacency {path.name}")
        return adj.tocoo()

    adj = adj.maximum(adj.T).tocsr()
    adj.setdiag(0)
    adj.eliminate_zeros()
    adj = adj + sp.eye(adj.shape[0], dtype=np.float32, format="csr")
    degree = np.asarray(adj.sum(axis=1)).reshape(-1)
    degree[degree <= 0] = 1.0
    inv_sqrt = np.power(degree, -0.5).astype(np.float32)
    d_inv = sp.diags(inv_sqrt)
    adj_norm = (d_inv @ adj @ d_inv).astype(np.float32).tocoo()
    print(f"{split}: normalized raw adjacency {path.name} with self-loops")
    return adj_norm


def scipy_to_torch_sparse(adj: sp.spmatrix, device: torch.device) -> torch.Tensor:
    coo = adj.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack([coo.row, coo.col]).astype(np.int64))
    values = torch.from_numpy(coo.data.astype(np.float32))
    return torch.sparse_coo_tensor(
        indices.to(device), values.to(device), size=coo.shape, device=device
    ).coalesce()


def sparse_mps_autograd_works() -> bool:
    """Test sparse.mm and backward support before using MPS for the GCN."""
    try:
        device = torch.device("mps")
        idx = torch.tensor([[0, 1], [0, 1]], dtype=torch.long, device=device)
        values = torch.ones(2, dtype=torch.float32, device=device)
        a = torch.sparse_coo_tensor(idx, values, (2, 2), device=device).coalesce()
        x = torch.randn(2, 3, dtype=torch.float32, device=device, requires_grad=True)
        torch.sparse.mm(a, x).sum().backward()
        torch.mps.synchronize()
        return x.grad is not None
    except Exception as exc:
        print(f"MPS sparse GCN operation unavailable ({type(exc).__name__}: {exc}).")
        return False


## 5. Define the GCN and evaluation metrics

There are two graph-convolution layers, followed by a one-logit mortality classifier. The model returns logits; `BCEWithLogitsLoss` and sigmoid are applied later.


In [19]:
class GCN(nn.Module):
    """Two graph-convolution layers followed by a mortality logit head."""
    def __init__(self, input_dim: int, hidden_dim: int = 128, dropout: float = 0.30):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h = torch.sparse.mm(adj, x)
        h = F.relu(self.fc1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = torch.sparse.mm(adj, h)
        h = F.relu(self.fc2(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h).squeeze(-1)


def metrics_at_threshold(y_true: np.ndarray, prob: np.ndarray, threshold: float) -> dict:
    pred = (prob >= threshold).astype(np.int64)
    return {
        "threshold": float(threshold),
        "auroc": float(roc_auc_score(y_true, prob)) if len(np.unique(y_true)) == 2 else None,
        "auprc": float(average_precision_score(y_true, prob)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, pred, labels=[0, 1]).tolist(),
    }


def best_f1_threshold(y_true: np.ndarray, prob: np.ndarray) -> float:
    """Select optimal F1 threshold using vectorized precision-recall curve on validation labels only."""
    precision, recall, thresholds = precision_recall_curve(y_true, prob)
    f1_scores = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1_scores))
    return float(thresholds[best_idx])


## 6. Load the three splits and check graph/embedding alignment

This cell reads the data, aligns patient IDs to graph order, and checks that each adjacency has exactly one row/column per patient.


In [20]:
set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
adjs: dict[str, sp.coo_matrix] = {}

print("Loading attention-fused embeddings and patient graphs...")
for split in ("train", "val", "test"):
    x, ids, y = load_embedding_file(split)
    x, ids, y = align_to_graph_order(split, x, ids, y)
    adj = load_adjacency(split)
    if adj.shape[0] != len(x):
        raise ValueError(
            f"{split}: adjacency has {adj.shape[0]} nodes but embeddings have {len(x)}. "
            "The graph and embedding patient order/selection must match exactly."
        )
    data[split] = (x, ids, y)
    adjs[split] = adj
    print(
        f"{split}: nodes={len(x):,}, adjacency entries={adj.nnz:,}, features={x.shape[1]}, "
        f"mortality={int(y.sum()):,} ({y.mean():.2%})"
    )

x_train, _, y_train_np = data["train"]
if len(np.unique(y_train_np)) < 2:
    raise ValueError("Training labels must contain both mortality classes.")
if len(np.unique(data["val"][2])) < 2:
    raise ValueError("Validation labels must contain both classes.")

# Fit feature scaling on training embeddings only, then reuse it for val/test.
x_mean = x_train.mean(axis=0, keepdims=True)
x_std = x_train.std(axis=0, keepdims=True)
x_std[x_std < 1e-8] = 1.0
for split in ("train", "val", "test"):
    x, ids, y = data[split]
    data[split] = ((x - x_mean) / x_std, ids, y)

print("Feature scaling fitted on training embeddings only.")


Loading attention-fused embeddings and patient graphs...
train: embeddings aligned to graph-node patient order.
train: using pre-normalized adjacency adjacency_norm_train.npz
train: nodes=153,831, adjacency entries=1,301,989, features=64, mortality=13,634 (8.86%)
val: embeddings aligned to graph-node patient order.
val: using pre-normalized adjacency adjacency_norm_val.npz
val: nodes=24,186, adjacency entries=202,504, features=64, mortality=2,289 (9.46%)
test: embeddings aligned to graph-node patient order.
test: using pre-normalized adjacency adjacency_norm_test.npz
test: nodes=22,809, adjacency entries=192,001, features=64, mortality=2,081 (9.12%)
Feature scaling fitted on training embeddings only.


In [21]:
print("PROJECT ROOT:", PROJECT_ROOT)

for split in ("train", "val", "test"):
    x, patient_ids, labels = data[split]
    adj = adjs[split]

    print(f"\n{split.upper()}")
    print(f"Embedding shape: {x.shape}")
    print(f"Patient IDs:     {len(patient_ids):,}")
    print(f"Labels:          {len(labels):,}")
    print(f"Graph shape:     {adj.shape}")
    print(f"Graph entries:   {adj.nnz:,}")
    print(f"Positive cases:  {int(labels.sum()):,}")
    print(f"Negative cases:  {int((labels == 0).sum()):,}")

    assert x.shape[0] == len(patient_ids) == len(labels)
    assert adj.shape == (len(labels), len(labels))

PROJECT ROOT: /Users/aryandwivedi/Desktop/Project Hail Minor

TRAIN
Embedding shape: (153831, 64)
Patient IDs:     153,831
Labels:          153,831
Graph shape:     (153831, 153831)
Graph entries:   1,301,989
Positive cases:  13,634
Negative cases:  140,197

VAL
Embedding shape: (24186, 64)
Patient IDs:     24,186
Labels:          24,186
Graph shape:     (24186, 24186)
Graph entries:   202,504
Positive cases:  2,289
Negative cases:  21,897

TEST
Embedding shape: (22809, 64)
Patient IDs:     22,809
Labels:          22,809
Graph shape:     (22809, 22809)
Graph entries:   192,001
Positive cases:  2,081
Negative cases:  20,728


## 7. Select device and create tensors/model

The notebook prefers CUDA when available, otherwise tests sparse GCN operations on MPS, and falls back to CPU only if needed.


In [22]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and sparse_mps_autograd_works():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device selected for GCN: {device}")
if device.type == "cpu" and torch.backends.mps.is_available():
    print("MPS is present, but sparse adjacency operations failed the support test; using CPU.")

tensors: dict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}
for split in ("train", "val", "test"):
    x_np, _, y_np = data[split]
    x_t = torch.tensor(x_np, dtype=torch.float32, device=device)
    y_t = torch.tensor(y_np, dtype=torch.float32, device=device)
    a_t = scipy_to_torch_sparse(adjs[split], device)
    tensors[split] = (x_t, y_t, a_t)

model = GCN(input_dim=tensors["train"][0].shape[1], hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

n_pos = float(y_train_np.sum())
n_neg = float(len(y_train_np) - y_train_np.sum())
if n_pos <= 0:
    raise ValueError("No positive mortality cases in training split.")
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training positive weight: {n_neg / n_pos:.4f}")


Device selected for GCN: mps
Model parameters: 24,961
Training positive weight: 10.2829


In [23]:
for split, (x, y, adj) in {
    "TRAIN": tensors["train"],
    "VAL": tensors["val"],
    "TEST": tensors["test"],
}.items():

    print(
        f"{split}: "
        f"features={tuple(x.shape)}, "
        f"labels={tuple(y.shape)}, "
        f"nodes={adj.shape[0]}, "
        f"edges={adj._nnz()}"
    )

TRAIN: features=(153831, 64), labels=(153831,), nodes=153831, edges=1301989
VAL: features=(24186, 64), labels=(24186,), nodes=24186, edges=202504
TEST: features=(22809, 64), labels=(22809,), nodes=22809, edges=192001


## 8. Train with validation AUPRC and early stopping

Each epoch performs one full-batch update on the training graph. After each update, validation AUPRC is evaluated. The best model state is kept; test data is not used in this cell.


In [24]:
# Set RECOMPUTE_HPO = True to retrain all 12 HPO trials (~15 mins).
# Set False to instantly load precomputed validated results from disk.
RECOMPUTE_HPO = False

import itertools
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score

# Reuse your already-loaded train and validation data.
xtr, ytr, atr = tensors["train"]
xval, yval, aval = tensors["val"]
yval_np = data["val"][2]

n_pos = float(y_train_np.sum())
n_neg = float(len(y_train_np) - n_pos)

checkpoint_file = OUTPUT_DIR / "gcn_attention.pt"
hpo_file = OUTPUT_DIR / "hpo_results.json"

if not RECOMPUTE_HPO and checkpoint_file.exists() and hpo_file.exists():
    print("Loading precomputed HPO results and best checkpoint from disk...")
    results = json.loads(hpo_file.read_text())
    best_config = results[0]
    best_ap = float(best_config["validation_auprc"])
    best_epoch = int(best_config["best_epoch"])
    
    saved_cp = torch.load(checkpoint_file, map_location=device, weights_only=False)
    best_state = saved_cp["model_state_dict"]
    HIDDEN_DIM = int(saved_cp.get("hidden_dim", best_config["hidden_dim"]))
    DROPOUT = float(saved_cp.get("dropout", best_config["dropout"]))
    
    model = GCN(
        input_dim=xtr.shape[1],
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ).to(device)
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        best_val_prob = torch.sigmoid(model(xval, aval)).detach().cpu().numpy()
        
    print(f"Loaded best checkpoint: trial={best_config['trial']}, epoch={best_epoch}, Val AUPRC={best_ap:.5f}")
else:
    N_TRIALS = 12
    HPO_EPOCHS = 80
    HPO_PATIENCE = 10

    # Reuse your already-loaded train and validation data.
    xtr, ytr, atr = tensors["train"]
    xval, yval, aval = tensors["val"]
    yval_np = data["val"][2]

    n_pos = float(y_train_np.sum())
    n_neg = float(len(y_train_np) - n_pos)

    if n_pos == 0 or n_neg == 0:
        raise ValueError("Training data must contain both classes.")

    pos_weight = torch.tensor(
        [n_neg / n_pos], dtype=torch.float32, device=device
    )

    # Define the search space.
    search_space = list(itertools.product(
        [64, 128, 256],       # hidden dimension
        [0.20, 0.35, 0.50],   # dropout
        [1e-4, 3e-4, 1e-3],   # learning rate
        [1e-5, 1e-4, 5e-4],   # weight decay
    ))

    rng = random.Random(SEED)
    configs = rng.sample(search_space, N_TRIALS)

    best_ap = -1.0
    best_state = None
    best_val_prob = None
    best_epoch = 0
    best_config = None
    results = []

    for trial, (hidden_dim, dropout, lr, wd) in enumerate(configs, start=1):
        print(f"\nTrial {trial}/{N_TRIALS}")
        print(
            f"hidden={hidden_dim}, dropout={dropout}, "
            f"lr={lr}, weight_decay={wd}"
        )

        # Reset initialization for reproducibility.
        set_seed(SEED)

        trial_model = GCN(
            input_dim=xtr.shape[1],
            hidden_dim=hidden_dim,
            dropout=dropout,
        ).to(device)

        optimizer = torch.optim.Adam(
            trial_model.parameters(),
            lr=lr,
            weight_decay=wd,
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        trial_best_ap = -1.0
        trial_best_epoch = 0
        trial_best_state = None
        trial_best_prob = None
        patience_counter = 0

        for epoch in range(1, HPO_EPOCHS + 1):
            trial_model.train()
            optimizer.zero_grad(set_to_none=True)

            logits = trial_model(xtr, atr)
            loss = criterion(logits, ytr)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                trial_model.parameters(), max_norm=5.0
            )
            optimizer.step()

            trial_model.eval()
            with torch.no_grad():
                val_prob = torch.sigmoid(
                    trial_model(xval, aval)
                ).detach().cpu().numpy()

            val_ap = float(
                average_precision_score(yval_np, val_prob)
            )

            if val_ap > trial_best_ap + 1e-6:
                trial_best_ap = val_ap
                trial_best_epoch = epoch
                trial_best_prob = val_prob.copy()
                trial_best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in trial_model.state_dict().items()
                }
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= HPO_PATIENCE:
                break

        print(
            f"Best validation AUPRC={trial_best_ap:.5f} "
            f"at epoch {trial_best_epoch}"
        )

        results.append({
            "trial": trial,
            "hidden_dim": hidden_dim,
            "dropout": dropout,
            "learning_rate": lr,
            "weight_decay": wd,
            "best_epoch": trial_best_epoch,
            "validation_auprc": trial_best_ap,
        })

        if trial_best_ap > best_ap:
            best_ap = trial_best_ap
            best_epoch = trial_best_epoch
            best_state = trial_best_state
            best_val_prob = trial_best_prob
            best_config = {
                "hidden_dim": hidden_dim,
                "dropout": dropout,
                "learning_rate": lr,
                "weight_decay": wd,
            }

    # Restore the best configuration and model weights.
    HIDDEN_DIM = best_config["hidden_dim"]
    DROPOUT = best_config["dropout"]
    LEARNING_RATE = best_config["learning_rate"]
    WEIGHT_DECAY = best_config["weight_decay"]

    model = GCN(
        input_dim=xtr.shape[1],
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ).to(device)
    model.load_state_dict(best_state)

    # Save a record of all trials.
    results.sort(key=lambda r: r["validation_auprc"], reverse=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "hpo_results.json").write_text(
        json.dumps(results, indent=2)
    )

    print("\nBEST GCN CONFIGURATION")
    print(json.dumps(best_config, indent=2))
    print(f"Best validation AUPRC: {best_ap:.5f}")
    print(f"Best epoch: {best_epoch}")
    print(f"Results saved to: {OUTPUT_DIR / 'hpo_results.json'}")


Loading precomputed HPO results and best checkpoint from disk...
Loaded best checkpoint: trial=8, epoch=36, Val AUPRC=0.53712


## 9. Choose threshold on validation, then evaluate test once

The classification threshold is selected by maximizing validation F1. The same threshold is then applied to the test predictions.


In [25]:
threshold = best_f1_threshold(yval_np, best_val_prob)
val_metrics = metrics_at_threshold(yval_np, best_val_prob, threshold)

model.eval()
xtest, ytest, atest = tensors["test"]
with torch.no_grad():
    test_prob = torch.sigmoid(model(xtest, atest)).detach().cpu().numpy()

test_metrics = metrics_at_threshold(data["test"][2], test_prob, threshold)

print(f"Validation-selected threshold: {threshold:.6f}")
print("Validation metrics:")
print(json.dumps(val_metrics, indent=2))
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))


Validation-selected threshold: 0.644971
Validation metrics:
{
  "threshold": 0.6449713110923767,
  "auroc": 0.8813065152145156,
  "auprc": 0.5382956269641411,
  "accuracy": 0.900645001240387,
  "balanced_accuracy": 0.7434639933141047,
  "precision": 0.4783269961977186,
  "recall": 0.5495849716033202,
  "f1": 0.511486074405367,
  "confusion_matrix": [
    [
      20525,
      1372
    ],
    [
      1031,
      1258
    ]
  ]
}
Test metrics:
{
  "threshold": 0.6449713110923767,
  "auroc": 0.8039318819014772,
  "auprc": 0.39312222078622067,
  "accuracy": 0.8173527993335964,
  "balanced_accuracy": 0.693087392576714,
  "precision": 0.2596264699100761,
  "recall": 0.5410860163382989,
  "f1": 0.35088812714241197,
  "confusion_matrix": [
    [
      17517,
      3211
    ],
    [
      955,
      1126
    ]
  ]
}


## 10. Save checkpoint, metrics, and test predictions

The output folder is `checkpoints/fusion/gnn_attention/`.


In [26]:
checkpoint_path = OUTPUT_DIR / "gcn_attention.pt"
metrics_path = OUTPUT_DIR / "metrics.json"
predictions_path = OUTPUT_DIR / "test_predictions.npz"

# Save the selected model and training-only scaling parameters.
torch.save(
    {
        "model_state_dict": best_state,
        "input_dim": int(xtr.shape[1]),
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "threshold": threshold,
        "best_epoch": best_epoch,
        "feature_mean": x_mean.astype(np.float32),
        "feature_std": x_std.astype(np.float32),
        "model_name": "two_layer_sparse_gcn_attention_fusion",
    },
    checkpoint_path,
)

result = {
    "model": "two_layer_sparse_gcn",
    "fusion": "attention",
    "device": str(device),
    "best_epoch_by_validation_auprc": int(best_epoch),
    "best_validation_auprc": float(best_ap),
    "validation_metrics_at_validation_selected_threshold": val_metrics,
    "test_metrics_at_same_threshold": test_metrics,
    "threshold_selected_on_validation_only": float(threshold),
    "counts": {
        split: {
            "patients": int(len(data[split][2])),
            "positive": int(data[split][2].sum()),
            "negative": int(len(data[split][2]) - data[split][2].sum()),
            "adjacency_entries": int(adjs[split].nnz),
        }
        for split in ("train", "val", "test")
    },
    "notes": [
        "GCN weights selected using validation AUPRC.",
        "Classification threshold selected using validation F1 only.",
        "Feature standardization statistics fitted on training embeddings only.",
        "Test labels are not used for training, early stopping, or threshold selection.",
    ],
}
metrics_path.write_text(json.dumps(result, indent=2))
np.savez_compressed(
    predictions_path,
    patient_ids=data["test"][1],
    labels=data["test"][2],
    probabilities=test_prob,
)

print("\nGCN RESULTS")
print(f"Device: {device}")
print(f"Best epoch by validation AUPRC: {best_epoch}")
print(f"Validation AUPRC: {best_ap:.4f}")
print(f"Threshold selected on validation: {threshold:.6f}")
for name, value in test_metrics.items():
    print(f"test_{name}: {value}")
print(f"Checkpoint:   {checkpoint_path}")
print(f"Metrics:      {metrics_path}")
print(f"Predictions:  {predictions_path}")



GCN RESULTS
Device: mps
Best epoch by validation AUPRC: 36
Validation AUPRC: 0.5371
Threshold selected on validation: 0.644971
test_threshold: 0.6449713110923767
test_auroc: 0.8039318819014772
test_auprc: 0.39312222078622067
test_accuracy: 0.8173527993335964
test_balanced_accuracy: 0.693087392576714
test_precision: 0.2596264699100761
test_recall: 0.5410860163382989
test_f1: 0.35088812714241197
test_confusion_matrix: [[17517, 3211], [955, 1126]]
Checkpoint:   /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_attention/gcn_attention.pt
Metrics:      /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_attention/metrics.json
Predictions:  /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gnn_attention/test_predictions.npz
